In [ ]:
!pip install scikit-learn joblib pandas numpy

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving synthetic_augmented_data.csv to synthetic_augmented_data.csv


In [ ]:
import pandas as pd
import numpy as np
import re
import random
import json
import warnings
from collections import Counter
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score)

In [ ]:
df = pd.read_csv('/content/synthetic_augmented_data.csv')

In [ ]:
warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

In [ ]:
df['label'] = (df['output'] == 'dementia').astype(int)   # 1=dementia, 0=control
print(f"Dataset loaded: {len(df)} samples  |  dementia={df['label'].sum()}  control={(df['label']==0).sum()}")


Dataset loaded: 996 samples  |  dementia=512  control=484


In [ ]:
FILLERS = {'uh', 'um', 'er', 'ah', 'like', 'you know', 'i mean', 'well', 'so', 'right'}

def extract_features(text: str) -> dict:
    """
    Extract a rich set of linguistic features known to discriminate
    Alzheimer's speech from healthy controls in the Cookie Theft task.
    """
    text = str(text)
    words_raw = text.split()
    words = [w.lower().strip('.,!?;:') for w in words_raw if w.strip('.,!?;:')]
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if len(s.strip()) > 3]
    n_words = max(len(words), 1)
    n_sents = max(len(sentences), 1)

    # ── Lexical richness ────────────────────────────────────────────────
    unique_words = set(words)
    ttr = len(unique_words) / n_words                          # Type-token ratio
    # Honore's statistic (log-based, robust to length)
    hapax = sum(1 for w, c in Counter(words).items() if c == 1)
    honore = (100 * np.log(n_words)) / max(1 - hapax / max(len(unique_words), 1), 1e-9)
    # Brunet's index (lower = richer vocabulary)
    brunet = n_words ** (len(unique_words) ** -0.165)

    # ── Fluency / disfluency ────────────────────────────────────────────
    filler_count = sum(1 for w in words if w in FILLERS)
    filler_rate = filler_count / n_words
    # Repetitions: count consecutive duplicate words
    repetitions = sum(1 for i in range(1, len(words)) if words[i] == words[i-1])
    repetition_rate = repetitions / n_words
    # Revision markers
    revision_words = {'no', 'wait', 'i mean', 'actually', 'sorry'}
    revision_count = sum(1 for w in words if w in revision_words)

    # ── Syntactic ───────────────────────────────────────────────────────
    mean_sent_len = n_words / n_sents
    # Sentence length variance (AD speech has more irregular phrasing)
    sent_lens = [len(s.split()) for s in sentences]
    sent_len_var = np.var(sent_lens) if len(sent_lens) > 1 else 0

    # ── Semantic / content ──────────────────────────────────────────────
    # Cookie Theft specific content words — informational units (IUs)
    iu_words = {
        'cookie', 'jar', 'boy', 'girl', 'stool', 'fall', 'falling', 'tip',
        'mother', 'woman', 'dishes', 'washing', 'sink', 'overflow', 'water',
        'kitchen', 'window', 'curtain', 'outside', 'garden', 'summer',
        'reach', 'reaching', 'hand', 'take', 'taking', 'steal', 'stealing'
    }
    iu_count = sum(1 for w in words if w in iu_words)
    iu_density = iu_count / n_words

    # Pronoun rate — AD speakers use more pronouns (referential ambiguity)
    pronouns = {'he', 'she', 'it', 'they', 'him', 'her', 'them', 'this', 'that', 'there'}
    pronoun_rate = sum(1 for w in words if w in pronouns) / n_words

    # Verb usage — action verbs indicate scene description ability
    action_verbs = {'is', 'are', 'get', 'getting', 'wash', 'washing', 'fall', 'falling',
                    'take', 'taking', 'reach', 'reaching', 'tip', 'tipping', 'run', 'running'}
    verb_rate = sum(1 for w in words if w in action_verbs) / n_words

    # ── Structural ──────────────────────────────────────────────────────
    word_lengths = [len(w) for w in words]
    mean_word_len = np.mean(word_lengths)
    # Conjunction rate — complex sentences
    conjunctions = {'and', 'but', 'because', 'while', 'although', 'since', 'when', 'if'}
    conj_rate = sum(1 for w in words if w in conjunctions) / n_words
    # Negation
    neg_rate = sum(1 for w in words if w in {"not", "no", "never", "don't", "doesn't", "isn't"}) / n_words

    return {
        'n_words': n_words,
        'n_sentences': n_sents,
        'ttr': ttr,
        'honore': honore,
        'brunet': brunet,
        'filler_rate': filler_rate,
        'repetition_rate': repetition_rate,
        'revision_count': revision_count,
        'mean_sent_len': mean_sent_len,
        'sent_len_var': sent_len_var,
        'iu_count': iu_count,
        'iu_density': iu_density,
        'pronoun_rate': pronoun_rate,
        'verb_rate': verb_rate,
        'mean_word_len': mean_word_len,
        'conj_rate': conj_rate,
        'neg_rate': neg_rate,
        'unique_words': len(unique_words),
    }


# Extract features for all samples
features_list = [extract_features(t) for t in df['input']]
feat_df = pd.DataFrame(features_list)
print(f"\nFeatures extracted: {feat_df.shape[1]} features × {feat_df.shape[0]} samples")


# ─────────────────────────────────────────────
# 3. EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("EXPLORATORY DATA ANALYSIS")
print("="*60)

feat_df['label'] = df['label'].values
feat_df['output'] = df['output'].values

key_features = ['ttr', 'filler_rate', 'repetition_rate', 'iu_density',
                'pronoun_rate', 'mean_sent_len', 'brunet', 'verb_rate']

print(f"\n{'Feature':<22} {'Dementia':>10} {'Control':>10} {'Diff%':>8}")
print("-" * 55)
for feat in key_features:
    d_mean = feat_df[feat_df['label']==1][feat].mean()
    c_mean = feat_df[feat_df['label']==0][feat].mean()
    pct = ((d_mean - c_mean) / max(abs(c_mean), 1e-9)) * 100
    print(f"  {feat:<20} {d_mean:>10.4f} {c_mean:>10.4f} {pct:>+8.1f}%")

feat_df = feat_df.drop(columns=['label', 'output'])


# ─────────────────────────────────────────────
# 4. SYNTHETIC DATA AUGMENTATION (EDA-style)
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("SYNTHETIC DATA AUGMENTATION")
print("="*60)

# Synonym dictionary for Cookie Theft domain (no NLTK needed)
SYNONYMS = {
    'kitchen': ['room', 'household area'],
    'mother': ['woman', 'lady', 'mom'],
    'boy': ['child', 'kid', 'young one'],
    'girl': ['child', 'kid', 'young one'],
    'cookie': ['biscuit', 'treat', 'snack'],
    'jar': ['container', 'pot', 'tin'],
    'stool': ['step stool', 'small ladder'],
    'sink': ['basin', 'washbasin'],
    'dishes': ['plates', 'crockery', 'utensils'],
    'washing': ['cleaning', 'rinsing'],
    'falling': ['tipping', 'toppling', 'about to fall'],
    'window': ['opening', 'glass pane'],
    'curtain': ['drape', 'blind'],
    'water': ['liquid'],
    'overflow': ['spill over', 'run over'],
    'reach': ['grab', 'get'],
    'see': ['notice', 'observe', 'look at'],
}

def synonym_replace(text: str, n: int = 2) -> str:
    """Replace n random words with domain synonyms."""
    words = text.split()
    indices = list(range(len(words)))
    random.shuffle(indices)
    replaced = 0
    for i in indices:
        w = words[i].lower().strip('.,!?')
        if w in SYNONYMS and replaced < n:
            words[i] = random.choice(SYNONYMS[w])
            replaced += 1
    return ' '.join(words)

def random_insertion(text: str, n: int = 1) -> str:
    """Insert filler words at random positions (mimics AD speech patterns)."""
    fillers = ['uh', 'um', 'er', 'well', 'you know']
    words = text.split()
    for _ in range(n):
        pos = random.randint(0, len(words))
        words.insert(pos, random.choice(fillers))
    return ' '.join(words)

def random_deletion(text: str, p: float = 0.05) -> str:
    """Randomly delete words with probability p."""
    words = text.split()
    if len(words) <= 5:
        return text
    return ' '.join(w for w in words if random.random() > p)

def swap_sentences(text: str) -> str:
    """Swap two adjacent sentences — disrupts narrative coherence."""
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sents) < 2:
        return text
    i = random.randint(0, len(sents) - 2)
    sents[i], sents[i+1] = sents[i+1], sents[i]
    return ' '.join(sents)

def augment_sample(text: str, label: int, n_aug: int = 2) -> list:
    """
    Apply EDA augmentation. For 'dementia' samples, we apply
    insertion more heavily to preserve disfluency signature.
    For 'control', we use synonym replacement and deletion.
    """
    augmented = []
    ops_dementia = [
        lambda t: random_insertion(t, n=random.randint(1,3)),
        lambda t: synonym_replace(t, n=1),
        lambda t: random_deletion(t, p=0.04),
    ]
    ops_control = [
        lambda t: synonym_replace(t, n=random.randint(2,4)),
        lambda t: random_deletion(t, p=0.03),
        lambda t: swap_sentences(t),
    ]
    ops = ops_dementia if label == 1 else ops_control
    for _ in range(n_aug):
        op = random.choice(ops)
        augmented.append(op(text))
    return augmented

# Generate synthetic samples — augment only training-split portion
# (We'll do a proper train/test split in evaluation, but generate
#  augmented pool here for use during cross-validation.)

aug_rows = []
for _, row in df.iterrows():
    aug_texts = augment_sample(row['input'], row['label'], n_aug=2)
    for t in aug_texts:
        aug_rows.append({'input': t, 'output': row['output'], 'label': row['label']})

aug_df = pd.DataFrame(aug_rows)
aug_features = pd.DataFrame([extract_features(t) for t in aug_df['input']])

print(f"  Original samples : {len(df)}")
print(f"  Augmented samples: {len(aug_df)}")
print(f"  Total pool       : {len(df) + len(aug_df)}")
print(f"  Aug dementia     : {(aug_df['label']==1).sum()}")
print(f"  Aug control      : {(aug_df['label']==0).sum()}")


# ─────────────────────────────────────────────
# 5. MODEL TRAINING & EVALUATION
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("MODEL EVALUATION (5-Fold Stratified Cross-Validation)")
print("="*60)

X_orig = feat_df.values
y_orig = df['label'].values

# Full augmented dataset
X_aug_only = aug_features.values
y_aug_only = aug_df['label'].values
X_full = np.vstack([X_orig, X_aug_only])
y_full = np.concatenate([y_orig, y_aug_only])

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(C=1.0, kernel='rbf', probability=True, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=8,
                                       random_state=42, n_jobs=-1))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=150, learning_rate=0.08,
                                            max_depth=4, random_state=42))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1', 'roc_auc']

results = {}

for name, model in models.items():
    # Without augmentation
    cv_orig = cross_validate(model, X_orig, y_orig, cv=cv, scoring=scoring, n_jobs=-1)
    # With augmentation
    cv_aug  = cross_validate(model, X_full,  y_full,  cv=cv, scoring=scoring, n_jobs=-1)

    results[name] = {
        'orig_acc':  cv_orig['test_accuracy'].mean(),
        'orig_f1':   cv_orig['test_f1'].mean(),
        'orig_auc':  cv_orig['test_roc_auc'].mean(),
        'aug_acc':   cv_aug['test_accuracy'].mean(),
        'aug_f1':    cv_aug['test_f1'].mean(),
        'aug_auc':   cv_aug['test_roc_auc'].mean(),
    }

    print(f"\n  {name}")
    print(f"    Without augmentation → Acc: {results[name]['orig_acc']:.3f}  "
          f"F1: {results[name]['orig_f1']:.3f}  AUC: {results[name]['orig_auc']:.3f}")
    print(f"    With augmentation    → Acc: {results[name]['aug_acc']:.3f}  "
          f"F1: {results[name]['aug_f1']:.3f}  AUC: {results[name]['aug_auc']:.3f}")
    delta = results[name]['aug_auc'] - results[name]['orig_auc']
    print(f"    AUC improvement      → {delta:+.3f}")


# ─────────────────────────────────────────────
# 6. FEATURE IMPORTANCE (best model)
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("FEATURE IMPORTANCE — Random Forest (augmented data)")
print("="*60)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=8,
                                   random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_full, y_full)
importances = rf_pipeline.named_steps['clf'].feature_importances_
feat_names = list(feat_df.columns)
imp_pairs = sorted(zip(feat_names, importances), key=lambda x: x[1], reverse=True)

print(f"\n  {'Feature':<22} Importance")
print("  " + "-"*38)
for fname, imp in imp_pairs:
    bar = '█' * int(imp * 200)
    print(f"  {fname:<22} {imp:.4f}  {bar}")


# ─────────────────────────────────────────────
# 7. SAVE RESULTS + AUGMENTED DATASET
# ─────────────────────────────────────────────
aug_df.to_csv('synthetic_augmented_data.csv', index=False)

summary = {
    'dataset_info': {
        'original_samples': int(len(df)),
        'dementia': int(df['label'].sum()),
        'control': int((df['label']==0).sum()),
        'augmented_samples': int(len(aug_df)),
        'total_pool': int(len(df) + len(aug_df)),
        'n_features': int(feat_df.shape[1]),
    },
    'model_results': {
        name: {k: round(v, 4) for k, v in vals.items()}
        for name, vals in results.items()
    },
    'feature_importance': {
        fname: round(float(imp), 4) for fname, imp in imp_pairs
    }
}
aug_df.to_csv('synthetic_augmented_data.csv', index=False)

with open('pipeline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*60)
print("FILES SAVED")
print("  /mnt/user-data/outputs/synthetic_augmented_data.csv")
print("  /mnt/user-data/outputs/pipeline_summary.json")
print("="*60)
import joblib

# Train final model on FULL augmented dataset
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42,
        n_jobs=-1
    ))
])

final_model.fit(X_full, y_full)

# Save model
joblib.dump(final_model, "alzheimers_model.pkl")

print("✅ Model saved as alzheimers_model.pkl")


Features extracted: 18 features × 996 samples

EXPLORATORY DATA ANALYSIS

Feature                  Dementia    Control    Diff%
-------------------------------------------------------
  ttr                      0.6334     0.6399     -1.0%
  filler_rate              0.0214     0.0101   +111.7%
  repetition_rate          0.0147     0.0049   +197.8%
  iu_density               0.1202     0.1398    -14.0%
  pronoun_rate             0.0617     0.0460    +34.2%
  mean_sent_len            8.5536     9.0023     -5.0%
  brunet                   9.9528    10.0777     -1.2%
  verb_rate                0.0704     0.0917    -23.2%

SYNTHETIC DATA AUGMENTATION
  Original samples : 996
  Augmented samples: 1992
  Total pool       : 2988
  Aug dementia     : 1024
  Aug control      : 968

MODEL EVALUATION (5-Fold Stratified Cross-Validation)

  Logistic Regression
    Without augmentation → Acc: 0.772  F1: 0.774  AUC: 0.858
    With augmentation    → Acc: 0.789  F1: 0.791  AUC: 0.883
    AUC improvemen